In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Qt5Agg') # Or 'TkAgg' depending on your system configuration
import matplotlib.pyplot as plt
import keras
import tensorflow as tf
import matplotlib.cm as cm


### Load Data

In [ ]:
img_path = keras.utils.get_file(fname="cat.jpg", origin="https://img-datasets.s3.amazonaws.com/cat.jpg")

def get_img_array(img_path, target_size):
    img = keras.utils.load_img(img_path, target_size=target_size)
    array = keras.utils.img_to_array(img)
    array = np.expand_dims(array, axis=0)
    return array

img_tensor = get_img_array(img_path, target_size=(180, 180))
img_tensor

In [ ]:
plt.axis("off")
plt.imshow(img_tensor[0].astype("uint8"))
plt.show()

### Load Model

In [ ]:
model = keras.models.load_model("convnet_from_scratch_with_augmentation_best.keras")
model.summary()

### Visualize Model Activation

In [ ]:
layer_outputs = []
layer_names = []
model.layers

In [ ]:
for layer in model.layers:
    if isinstance(layer, (keras.layers.Conv2D, keras.layers.MaxPooling2D)):
        layer_outputs.append(layer.output)
        layer_names.append(layer.name)

activation_model = keras.Model(inputs=model.input, outputs=layer_outputs)
activation_model.summary()

In [ ]:
activations = activation_model.predict(img_tensor)
plt.matshow(activations[0][0, :, :, 2], cmap="viridis")

In [ ]:

plt.matshow(activations[0][0, :, :, 3], cmap="viridis")

In [ ]:
images_per_row = 8
for layer_name, layer_activation in zip(layer_names, activations):
    n_features = layer_activation.shape[-1]
    size = layer_activation.shape[1]
    n_cols = n_features // images_per_row
    display_grid = np.zeros(((size + 1) * n_cols - 1, images_per_row * (size + 1) - 1))
    for col in range(n_cols):
        for row in range(images_per_row):
            channel_index = col * images_per_row + row
            channel_image = layer_activation[0, :, :, channel_index].copy()
            if channel_image.sum() != 0:
                channel_image -= channel_image.mean()
                channel_image /= channel_image.std()
                channel_image *= 64
                channel_image += 128
            channel_image = np.clip(channel_image, 0, 255).astype("uint8")

            display_grid[col * (size + 1) : (col + 1) * size + col, row * (size + 1) : (row + 1) * size + row] = channel_image
    scale = 1.0 / size
    plt.figure(figsize=(scale * display_grid.shape[1], scale * display_grid.shape[0]))
    plt.title(layer_name)
    plt.grid(False)
    plt.axis("off")
    plt.imshow(display_grid, aspect="auto", cmap="viridis")


### Visualizing Model Filters

In [ ]:
layer_outputs = []
layer_names = []
model.layers

In [ ]:
for layer in model.layers:
    if isinstance(layer, (keras.layers.Conv2D, keras.layers.MaxPooling2D)):
        layer_outputs.append(layer.output)
        layer_names.append(layer.name)

feature_extractor = keras.Model(inputs=model.input, outputs=layer_outputs)
feature_extractor.summary()

In [ ]:
def compute_loss(image, filter_index, layer_num=0):
    activation = feature_extractor(image)
    filter_activation = activation[layer_num][:, :, :, filter_index]
    return keras.ops.mean(filter_activation)

@tf.function
def gradient_ascent_step(image, filter_index, learning_rate, layer_num=0):
    with tf.GradientTape() as tape:
        tape.watch(image)
        loss = compute_loss(image, filter_index, layer_num)
    grads = tape.gradient(loss, image)
    grads = keras.ops.normalize(grads)
    image += learning_rate * grads
    return image

In [ ]:
def generate_filter_pattern(filter_index, layer_num=0, img_width=180, img_height=180):
    iterations = 30
    learning_rate = 10.0
    image = keras.random.uniform(minval=0.4, maxval=0.6, shape=(1, img_width, img_height, 3))
    for i in range(iterations):
        image = gradient_ascent_step(image, filter_index, learning_rate, layer_num)
    return image[0]


def deprocess_image(image):
    image -= keras.ops.mean(image)
    image /= keras.ops.std(image)
    image *= 64
    image += 128
    image = keras.ops.clip(image, 0, 255)
    image = image[1:-1, 1:-1, :]
    image = keras.ops.cast(image, dtype="uint8")
    return keras.ops.convert_to_numpy(image).astype("uint8")

In [ ]:
plt.axis("off")
channel_image = deprocess_image(generate_filter_pattern(filter_index=2, layer_num=4))
plt.imshow(channel_image)

In [ ]:
images_per_row = 8
for layer_num, (layer_name, layer_activation) in enumerate(zip(layer_names, activations)):
    n_features = layer_activation.shape[-1]
    size = 178
    n_cols = n_features // images_per_row
    display_grid = np.zeros(((size + 1) * n_cols - 1, images_per_row * (size + 1) - 1, 3))
    for col in range(n_cols):
        for row in range(images_per_row):
            channel_index = col * images_per_row + row
            channel_image = deprocess_image(generate_filter_pattern(filter_index=channel_index, layer_num=layer_num))
            display_grid[col * (size + 1) : (col + 1) * size + col, row * (size + 1) : (row + 1) * size + row,:] = channel_image
    scale = 1.0 / size
    plt.figure(figsize=(scale * display_grid.shape[1], scale * display_grid.shape[0]))
    plt.title(layer_name)
    plt.grid(False)
    plt.axis("off")
    plt.imshow(display_grid.astype("uint8"), aspect="auto", cmap="viridis")

### Heatmap Input Image

In [ ]:
preds = model.predict(img_tensor)
preds

In [ ]:
last_conv_layer_name = None
for layer in reversed(model.layers):
    if "conv" in layer.name.lower():
        last_conv_layer_name = layer.name
        break
print(f"Using last convolutional layer: {last_conv_layer_name}")

In [ ]:
grad_model = keras.models.Model(
    inputs=model.inputs,
    outputs=[model.get_layer(last_conv_layer_name).output, model.output]
)
grad_model.summary()

In [ ]:
with tf.GradientTape() as tape:
    conv_outputs, predictions = grad_model(img_tensor)
    loss = predictions[0]

grads = tape.gradient(loss, conv_outputs)
pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

In [ ]:
conv_outputs = conv_outputs[0]
heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
heatmap = tf.squeeze(heatmap)

In [ ]:
heatmap = np.maximum(heatmap, 0)
heatmap /= np.max(heatmap)
plt.matshow(heatmap)

In [ ]:
img = keras.utils.load_img(img_path)
img = keras.utils.img_to_array(img)
heatmap = np.uint8(255 * heatmap)
jet = cm.get_cmap("jet")
jet_colors = jet(np.arange(256))[:, :3]
jet_heatmap = jet_colors[heatmap]
jet_heatmap = keras.utils.array_to_img(jet_heatmap)
jet_heatmap = jet_heatmap.resize((img.shape[1], img.shape[0]))
jet_heatmap = keras.utils.img_to_array(jet_heatmap)
superimposed_img = jet_heatmap * 0.4 + img
superimposed_img = keras.utils.array_to_img(superimposed_img)
plt.imshow(superimposed_img)

### Latent Space Viz

In [ ]:
validation_dataset = tf.keras.utils.image_dataset_from_directory(r'C:\Users\PRASHANTH N\PycharmProjects\MTechSem2\DLRL\BOOKS\1\chapter8\dogs_vs_cats_small\validation', image_size=(180, 180), batch_size=1000,shuffle=True)

for images, labels in validation_dataset.take(1):
    X_val = images.numpy()
    y_val = labels.numpy()
    break

print(f"Loaded X_val shape: {X_val.shape}")


In [ ]:
bottleneck_layer_name = model.layers[-2].name
feature_extractor = keras.Model(inputs=model.input, outputs=model.get_layer(bottleneck_layer_name).output)
latent_features = feature_extractor.predict(X_val)
if len(latent_features.shape) > 2:
    latent_features = latent_features.reshape(latent_features.shape[0], -1)
print(f"Extracted feature shape: {latent_features.shape}")

In [ ]:
from sklearn.manifold import TSNE
tsne = TSNE(
    n_components=2,
    perplexity=30.0,
    max_iter=1000,
    random_state=42
)
low_dim_features = tsne.fit_transform(latent_features)
print(f"2D features shape: {low_dim_features.shape}")

In [ ]:
plt.figure(figsize=(10, 8))
x = low_dim_features[:, 0]
y = low_dim_features[:, 1]
plt.scatter(x[y_val == 0], y[y_val == 0], c='orange', label='Cats', alpha=0.6, edgecolors='w')
plt.scatter(x[y_val == 1], y[y_val == 1], c='blue', label='Dogs', alpha=0.6, edgecolors='w')
plt.title('Latent Space Visualization of Cats vs. Dogs CNN')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:

tsne_3d = TSNE(
    n_components=3,
    perplexity=30.0,
    max_iter=1000,
    random_state=42
)
low_dim_features_3d = tsne_3d.fit_transform(latent_features)
print(f"3D features shape: {low_dim_features_3d.shape}")  # (N, 3)
x = low_dim_features_3d[:, 0]
y = low_dim_features_3d[:, 1]
z = low_dim_features_3d[:, 2]
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(x[y_val == 0], y[y_val == 0], z[y_val == 0], c='orange', label='Cats', alpha=0.6, edgecolors='w')
ax.scatter(x[y_val == 1], y[y_val == 1], z[y_val == 1], c='blue', label='Dogs', alpha=0.6, edgecolors='w')

ax.set_title('3D Latent Space Visualization')
ax.set_xlabel('t-SNE Dim 1')
ax.set_ylabel('t-SNE Dim 2')
ax.set_zlabel('t-SNE Dim 3')
ax.legend()
plt.show()